# Deep learning in practice: training fast on a GPU [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/x-datascience-datacamp/datacamp-master/blob/main/09_practical_deep_learning/01-deep_learning_in_practice.ipynb)

[Thomas Moreau](https://tommoral.github.io/about.html) and
[Pedro L. C. Rodrigues](https://plcrodrigues.github.io/)

---

Training a neural network is mostly about two things: getting a correct training
loop, and keeping the GPU busy. This notebook trains a small ResNet on CIFAR-10 and
goes from a slow, naive version to a version several times faster, measuring each
step.

**On Colab**, select a GPU first: *Runtime > Change runtime type > T4 GPU*.

## Table of contents

* [1 A minimal training loop](#loop)
    * [1.1 Data and model](#loop_data)
    * [1.2 Sanity checks](#loop_sanity)
    * [1.3 Training and evaluation](#loop_train)
* [2 Keeping the GPU busy](#gpu)
    * [2.1 Who is the bottleneck?](#gpu_bottleneck)
    * [2.2 Faster data loading](#gpu_data)
    * [2.3 Batch size](#gpu_batch)
    * [2.4 Mixed precision](#gpu_amp)
    * [2.5 Fused optimizer and `torch.compile`](#gpu_compile)
* [3 Learning rate and batch size](#lr)
* [4 More tips for practice](#tips)

In [ ]:
import os
import time
from itertools import islice

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, device: {device}")
if device == "cuda":
    print(torch.cuda.get_device_name())

`nvidia-smi` shows the GPU, its memory and its current load. Run
`watch -n 1 nvidia-smi` in a terminal while training to see whether the GPU is busy
(on Colab, the *Resources* panel shows the same).

In [ ]:
!nvidia-smi

# 1 A minimal training loop <a class="anchor" id="loop"></a> [↑](#Table-of-contents)

## 1.1 Data and model <a class="anchor" id="loop_data"></a> [↑](#Table-of-contents)

CIFAR-10 has 60,000 color images of 32x32 pixels in 10 classes. We keep 5,000 of the
training images for validation: the test set is used only once, at the very end.

The training images are augmented with random crops and flips. As usual in
`torchvision`, the augmentation is done image by image, on the CPU, when the image is
loaded.

In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_transform = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

cifar_train = torchvision.datasets.CIFAR10("data", train=True, download=True)
cifar_test = torchvision.datasets.CIFAR10("data", train=False, download=True)

perm = torch.randperm(50_000, generator=torch.Generator().manual_seed(0))
train_idx, val_idx = perm[:45_000], perm[45_000:]
train_set = Subset(
    torchvision.datasets.CIFAR10("data", train=True, transform=train_transform),
    train_idx,
)
val_set = Subset(
    torchvision.datasets.CIFAR10("data", train=True, transform=eval_transform), val_idx
)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, image, label in zip(axes.flat, cifar_train.data, cifar_train.targets):
    ax.imshow(image)
    ax.set_title(cifar_train.classes[label], fontsize=9)
    ax.axis("off")

The model is a ResNet-9: a few convolutions with batch normalization, two residual
blocks and a linear classifier. It is small (6.5M parameters) but reaches more than
90% accuracy in a few minutes of training.

In [ ]:
def conv_bn(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
    )


class Residual(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(conv_bn(channels, channels), conv_bn(channels, channels))

    def forward(self, x):
        return x + self.block(x)


def make_model():
    model = nn.Sequential(
        conv_bn(3, 64),
        conv_bn(64, 128), nn.MaxPool2d(2), Residual(128),
        conv_bn(128, 256), nn.MaxPool2d(2),
        conv_bn(256, 512), nn.MaxPool2d(2), Residual(512),
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),
        nn.Linear(512, 10),
    )
    nn.init.zeros_(model[-1].weight)  # uniform predictions at initialization
    return model.to(device)


model = make_model()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

## 1.2 Sanity checks <a class="anchor" id="loop_sanity"></a> [↑](#Table-of-contents)

Before any long training, two checks catch most bugs in a few seconds:

* **The initial loss.** With 10 balanced classes, an untrained network should predict
  about $1/10$ for each class, so the cross-entropy should be close to
  $\ln(10) \approx 2.3$. A much larger value means a bad initialization or a bug in the
  loss. With the default initialization, this network starts at about 3.6: its
  predictions are confidently wrong. This is why `make_model` sets the weights of the
  last layer to 0.
* **Overfitting a single batch.** The model must be able to learn one batch by heart.
  If the loss does not go to 0, the training loop is broken: wrong labels, a missing
  `optimizer.zero_grad()`, a learning rate far off...

In [ ]:
X, y = next(iter(DataLoader(train_set, batch_size=128, shuffle=True)))
X, y = X.to(device), y.to(device)

model = make_model()
print(f"Initial loss: {F.cross_entropy(model(X), y).item():.2f} "
      f"(ln(10) = {np.log(10):.2f})")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(50):
    loss = F.cross_entropy(model(X), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"Loss after 50 steps on the same batch: {loss.item():.4f}")

## 1.3 Training and evaluation <a class="anchor" id="loop_train"></a> [↑](#Table-of-contents)

The training loop is always the same five lines: forward, loss, `zero_grad`,
backward, step. We use `AdamW` (Adam with a correct weight decay): with a learning
rate around `1e-3`, it is a good default for most problems, and it needs much less
tuning than SGD.

The loop already has two hooks that we use in Part 2: mixed precision (the `scaler`
and `autocast`, disabled for now) and a learning rate scheduler.

In [ ]:
def train_epoch(model, loader, optimizer, scaler, scheduler=None):
    model.train()
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=scaler.is_enabled()):
            loss = F.cross_entropy(model(X), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
    return loss.item()


@torch.inference_mode()
def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device == "cuda"):
            correct += (model(X).argmax(dim=1) == y).sum()
        total += len(y)
    return correct.item() / total

Note the `model.train()` and `model.eval()` calls: batch normalization and dropout
behave differently during training and evaluation. `torch.inference_mode()` turns off
the gradient bookkeeping during evaluation.

We train for one epoch with a plain `DataLoader`.

In [ ]:
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=512)

model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
no_amp = torch.amp.GradScaler(enabled=False)

start = time.perf_counter()
loss = train_epoch(model, train_loader, optimizer, no_amp)
print(f"One epoch in {time.perf_counter() - start:.0f}s, last loss {loss:.2f}, "
      f"validation accuracy {accuracy(model, val_loader):.1%}")

# 2 Keeping the GPU busy <a class="anchor" id="gpu"></a> [↑](#Table-of-contents)

A GPU is fast only if it always has work to do. The right metric is the
**throughput**: the number of training images processed per second. We measure it on
a few training steps.

GPU operations are asynchronous: Python only queues them, and moves on. To time them,
we must wait for the GPU to finish with `torch.cuda.synchronize()`.

In [ ]:
def images_per_second(model, loader, use_amp=False, fused=False, n_steps=30):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, fused=fused)
    scaler = torch.amp.GradScaler(enabled=use_amp)
    train_epoch(model, islice(loader, 3), optimizer, scaler)  # warm-up
    torch.cuda.synchronize()
    start = time.perf_counter()
    train_epoch(model, islice(loader, n_steps), optimizer, scaler)
    torch.cuda.synchronize()
    return n_steps * loader.batch_size / (time.perf_counter() - start)


def data_images_per_second(loader, n_steps=30):
    """Throughput of the data loading alone, without any training."""
    start = time.perf_counter()
    for X, y in islice(loader, n_steps):
        X = X.to(device, non_blocking=True)
    torch.cuda.synchronize()
    return n_steps * loader.batch_size / (time.perf_counter() - start)


speeds = {}

## 2.1 Who is the bottleneck? <a class="anchor" id="gpu_bottleneck"></a> [↑](#Table-of-contents)

The training speed is limited by the slowest of the data loading and the computation.
Comparing the training throughput with the throughput of the data loading alone tells
which one it is.

In [ ]:
model = make_model()
speeds["DataLoader"] = images_per_second(model, train_loader)
print(f"Training:          {speeds['DataLoader']:6.0f} images/s")
print(f"Data loading only: {data_images_per_second(train_loader):6.0f} images/s")

The two numbers are close: the GPU spends most of its time waiting for the CPU, which
opens, crops and flips each image in Python, one at a time.

## 2.2 Faster data loading <a class="anchor" id="gpu_data"></a> [↑](#Table-of-contents)

**Several workers.** With `num_workers > 0`, the `DataLoader` prepares the next
batches in parallel processes while the GPU computes. `pin_memory=True` puts the
batches in page-locked memory, which makes the copy to the GPU faster and lets it
overlap with the computation (`non_blocking=True` in `train_epoch`).
`persistent_workers=True` keeps the workers alive between epochs.

In [ ]:
n_workers = min(os.cpu_count(), 8)
fast_loader = DataLoader(
    train_set, batch_size=128, shuffle=True, drop_last=True,
    num_workers=n_workers, pin_memory=True, persistent_workers=True,
)
speeds[f"DataLoader, {n_workers} workers"] = images_per_second(model, fast_loader)
print(f"{n_workers} workers: {speeds[f'DataLoader, {n_workers} workers']:.0f} images/s")

The gain depends on the number of CPU cores: Colab only has 2.

**The whole dataset on the GPU.** CIFAR-10 is 150 MB as `uint8`: it fits in GPU
memory many times over. We can then skip the `DataLoader` entirely: keep the images on
the GPU, draw the batches by indexing, and do the augmentation on the GPU, on the whole
batch at once. This works for any dataset that fits in GPU memory.

In [ ]:
MEAN_GPU = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
STD_GPU = torch.tensor(STD, device=device).view(1, 3, 1, 1)


def to_gpu(images, labels):
    images = torch.as_tensor(images).permute(0, 3, 1, 2)  # (N, H, W, C) -> (N, C, H, W)
    return images.to(device), torch.as_tensor(labels).to(device)


X_train, y_train = to_gpu(cifar_train.data[train_idx], np.array(cifar_train.targets)[train_idx])
X_val, y_val = to_gpu(cifar_train.data[val_idx], np.array(cifar_train.targets)[val_idx])
X_test, y_test = to_gpu(cifar_test.data, cifar_test.targets)


def augment(X):
    """Random horizontal flip and random crop with 4 pixels of padding, per image."""
    n = len(X)
    flip = torch.rand(n, 1, 1, 1, device=device) < 0.5
    X = torch.where(flip, X.flip(3), X)
    X = F.pad(X, (4, 4, 4, 4), mode="reflect")
    i, j = torch.randint(0, 9, (2, n, 1), device=device)
    rows = (i + torch.arange(32, device=device))[:, None, :, None]
    cols = (j + torch.arange(32, device=device))[:, None, None, :]
    batch = torch.arange(n, device=device)[:, None, None, None]
    channel = torch.arange(3, device=device)[None, :, None, None]
    return X[batch, channel, rows, cols]


class GPULoader:
    """Batches of images stored as one tensor on the GPU."""

    def __init__(self, images, labels, batch_size, train):
        self.images, self.labels = images, labels
        self.batch_size, self.train = batch_size, train

    def __len__(self):
        n = len(self.labels)
        return n // self.batch_size if self.train else -(-n // self.batch_size)

    def __iter__(self):
        n = len(self.labels)
        order = torch.randperm(n, device=device) if self.train else torch.arange(n, device=device)
        for k in range(len(self)):
            idx = order[k * self.batch_size:(k + 1) * self.batch_size]
            X = self.images[idx].float() / 255
            if self.train:
                X = augment(X)
            yield (X - MEAN_GPU) / STD_GPU, self.labels[idx]


gpu_loader = GPULoader(X_train, y_train, batch_size=128, train=True)
speeds["GPU loader"] = images_per_second(model, gpu_loader)
print(f"Training:          {speeds['GPU loader']:6.0f} images/s")
print(f"Data loading only: {data_images_per_second(gpu_loader):6.0f} images/s")

The data loading is now much faster than the training: the GPU never waits, and the
bottleneck is the computation itself.

## 2.3 Batch size <a class="anchor" id="gpu_batch"></a> [↑](#Table-of-contents)

A GPU has thousands of cores. With a small batch, many of them stay idle, and the
fixed cost of launching each operation dominates. Larger batches use the GPU better,
up to a point where it is saturated, and then the memory runs out.

In [ ]:
batch_sizes = [32, 64, 128, 256, 512, 1024]
throughputs, memories = [], []
for batch_size in batch_sizes:
    torch.cuda.reset_peak_memory_stats()
    loader = GPULoader(X_train, y_train, batch_size, train=True)
    throughputs.append(images_per_second(model, loader))
    memories.append(torch.cuda.max_memory_allocated() / 1e9)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(batch_sizes, throughputs, "o-")
ax1.set(xscale="log", xlabel="Batch size", ylabel="Images / s", title="Throughput")
ax2.plot(batch_sizes, memories, "o-")
ax2.set(xscale="log", xlabel="Batch size", ylabel="GB", title="Peak GPU memory")
ax1.set_xticks(batch_sizes, batch_sizes)
ax2.set_xticks(batch_sizes, batch_sizes)
plt.tight_layout()

Pick the smallest batch size that is close to the plateau. A larger batch does not
make the training faster, and as we see in Part 3, it changes the optimization.

In [ ]:
BATCH_SIZE = 512
gpu_loader = GPULoader(X_train, y_train, BATCH_SIZE, train=True)
speeds[f"GPU loader, batch {BATCH_SIZE}"] = images_per_second(model, gpu_loader)

## 2.4 Mixed precision <a class="anchor" id="gpu_amp"></a> [↑](#Table-of-contents)

By default, all computations are in 32-bit floats (`float32`). Recent GPUs have
*tensor cores* that compute much faster in 16 bits. With `torch.autocast`, the
operations that are safe in 16 bits (matrix products, convolutions) run in 16 bits,
and the others (reductions, the loss) stay in 32 bits.

* `float16` has a small range: small gradients round to 0. The `GradScaler`
  multiplies the loss by a large factor before the backward pass, and divides the
  gradients back before the step.
* `bfloat16` has the range of `float32`, and needs no scaler. Use it on recent GPUs
  (A100, H100). The T4 of Colab and the V100 do not support it.

Convolutions on tensor cores are also faster when the channels are the last dimension
in memory (`channels_last`).

In [ ]:
model = make_model().to(memory_format=torch.channels_last)
speeds["+ mixed precision"] = images_per_second(model, gpu_loader, use_amp=True)
print(f"Mixed precision: {speeds['+ mixed precision']:.0f} images/s")

## 2.5 Fused optimizer and `torch.compile` <a class="anchor" id="gpu_compile"></a> [↑](#Table-of-contents)

In eager mode (the default), PyTorch runs each operation as a separate GPU kernel, and
writes each intermediate result to memory. Small operations then waste most of their
time launching kernels and moving data.

**Fused optimizer.** The `AdamW` update is a dozen small operations per parameter
tensor. `fused=True` runs them all in one kernel.

In [ ]:
speeds["+ fused AdamW"] = images_per_second(model, gpu_loader, use_amp=True, fused=True)
print(f"Fused AdamW: {speeds['+ fused AdamW']:.0f} images/s")

**`torch.compile`** traces the model once, and generates optimized kernels that
*fuse* successive operations (e.g. a normalization and an activation), which saves
many memory round-trips.

The first call is slow, since it compiles. The compiled model is then reused as long
as the input shapes do not change: a different batch size triggers a new compilation.
This is why the training loader drops the last incomplete batch.

In [ ]:
compiled_model = torch.compile(model)

start = time.perf_counter()
images_per_second(compiled_model, gpu_loader, use_amp=True, fused=True, n_steps=1)
print(f"Compilation: {time.perf_counter() - start:.0f}s")

speeds["+ torch.compile"] = images_per_second(
    compiled_model, gpu_loader, use_amp=True, fused=True
)
print(f"Compiled: {speeds['+ torch.compile']:.0f} images/s")

The gain depends on the model. A ResNet spends most of its time in convolutions,
which are already calls to the optimized kernels of cuDNN: there is little left to
fuse, and the compiled version can even be slower. A transformer has many small
operations around its matrix products (layer norms, activations, attention softmax),
which `torch.compile` fuses. Let us compare with a small Vision Transformer (ViT), on
the same images.

In [ ]:
class ViT(nn.Module):
    """Cut the image in 4x4 patches, and feed them to a transformer encoder."""

    def __init__(self, dim=256, depth=6, heads=8):
        super().__init__()
        self.patches = nn.Conv2d(3, dim, kernel_size=4, stride=4)
        self.position = nn.Parameter(torch.zeros(1, 64, dim))
        layer = nn.TransformerEncoderLayer(
            dim, heads, 4 * dim, dropout=0.0, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 10))

    def forward(self, x):
        x = self.patches(x).flatten(2).transpose(1, 2) + self.position
        return self.head(self.encoder(x).mean(dim=1))


vit = ViT().to(device)
vit_eager = images_per_second(vit, gpu_loader, use_amp=True, fused=True)
vit_compiled = images_per_second(torch.compile(vit), gpu_loader, use_amp=True, fused=True)
print(f"ViT eager:    {vit_eager:6.0f} images/s")
print(f"ViT compiled: {vit_compiled:6.0f} images/s (x{vit_compiled / vit_eager:.1f})")

For transformers, `torch.compile` is one of the largest gains, and it is worth the
compilation time for any training longer than a few minutes. For our ResNet, we stay
in eager mode.

Let us summarize what we gained.

In [ ]:
names = list(speeds)
plt.figure(figsize=(6, 3))
plt.barh(names, [speeds[name] for name in names])
plt.gca().invert_yaxis()
plt.xlabel("Images / s")
plt.title(torch.cuda.get_device_name())
for name in names:
    print(f"{name:30s} {speeds[name]:7.0f} images/s  (x{speeds[name] / speeds['DataLoader']:.1f})")

<div class="alert alert-success">
    <p><b>Exercise</b>:</p>

Compare your numbers with the ones obtained on another GPU (for instance, those shown
in class on a V100).

1. Which steps give the largest gains on your GPU? Are they the same?
2. On your GPU, does `torch.compile` speed up the ResNet? And the ViT?

There is no universal recipe: the bottleneck depends on the hardware, the model and
the data. Always measure.

</div>

In [ ]:
# Your work.

# 3 Learning rate and batch size <a class="anchor" id="lr"></a> [↑](#Table-of-contents)

A gradient computed on a batch of size $B$ is a noisy estimate of the full gradient,
with a variance that decreases as $1/B$. A larger batch gives a more precise gradient,
so the optimizer can take larger steps, but it also makes fewer steps per epoch. To keep
the same progress per epoch, **the learning rate must grow with the batch size**:

* for SGD, linearly: $\eta \propto B$
  ([Goyal et al., 2017](https://arxiv.org/abs/1706.02677)),
* for Adam, as the square root: $\eta \propto \sqrt{B}$
  ([Malladi et al., 2022](https://arxiv.org/abs/2205.10287)).

These rules are a starting point, not a law: the learning rate must still be tuned.
They hold up to a *critical batch size*, beyond which larger batches no longer reduce
the number of steps needed
([McCandlish et al., 2018](https://arxiv.org/abs/1812.06162)): the extra samples are
wasted.

A large learning rate is unstable at the start of training, when the weights are
random. The usual schedule is a short **warm-up** (the learning rate grows linearly
from almost 0), followed by a slow decrease to 0 (here linear, often a cosine).
`OneCycleLR` implements it.

In [ ]:
def fit(batch_size, lr, epochs):
    model = make_model().to(memory_format=torch.channels_last)
    loader = GPULoader(X_train, y_train, batch_size, train=True)
    val_loader = GPULoader(X_val, y_val, 1000, train=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4, fused=True)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=epochs * len(loader), pct_start=0.2,
        anneal_strategy="linear",
    )
    scaler = torch.amp.GradScaler()

    history = []
    start = time.perf_counter()
    for epoch in range(epochs):
        train_epoch(model, loader, optimizer, scaler, scheduler)
        history.append((time.perf_counter() - start, accuracy(model, val_loader)))
    return model, np.array(history)


model, history = fit(BATCH_SIZE, lr=2e-3, epochs=10)
print(f"10 epochs in {history[-1, 0]:.0f}s, "
      f"validation accuracy {history[-1, 1]:.1%}")

Only now, with the model chosen, do we look at the test set, once.

In [ ]:
test_loader = GPULoader(X_test, y_test, 1000, train=False)
print(f"Test accuracy: {accuracy(model, test_loader):.1%}")

<div class="alert alert-success">
    <p><b>Exercise</b>:</p>

For $B \in \{128, 2048\}$, train for 4 epochs with each learning rate
$\eta \in \{10^{-3}, 2 \cdot 10^{-3}, 4 \cdot 10^{-3}, 8 \cdot 10^{-3}\}$, and plot the
final validation accuracy as a function of $\eta$, with one line per batch size.

1. Does the best learning rate grow with the batch size? By how much, compared with
   the $\sqrt{B}$ rule?
2. Is $B = 2048$ faster to train? Is it better? Remember the throughput plateau of
   Section 2.3.

</div>

In [ ]:
# Your work.

In [ ]:
# %load solutions/solution_1.py

# 4 More tips for practice <a class="anchor" id="tips"></a> [↑](#Table-of-contents)

**Do not train from scratch if you can avoid it.** For images, text or sound, start
from a pretrained model and fine-tune it, or train a linear model on its features.
Libraries such as [`timm`](https://huggingface.co/docs/timm) (images) and Hugging Face
[`transformers`](https://huggingface.co/docs/transformers) give access to thousands of
them:

```python
import timm
model = timm.create_model("resnet18", pretrained=True, num_classes=10)
```

**On tabular data, do not use a neural network first.** Gradient boosting and the
tabular foundation models of Day 3 are hard to beat.

**Tune the learning rate first.** It is the hyperparameter that matters most. Try a
log grid (`1e-4, 3e-4, 1e-3, 3e-3`), on short runs.

**Optimizers.** `AdamW` is a robust default. For transformers, recent optimizers
such as Muon or SOAP train faster: the
[benchopt nanoGPT benchmark](https://github.com/benchopt/benchmark_nanogpt) compares
them on the same GPT model.

**When the batch does not fit in memory**, accumulate the gradients of several small
batches before each step. This gives the same gradient as one large batch:

```python
for k, (X, y) in enumerate(loader):
    loss = F.cross_entropy(model(X), y) / n_accumulation
    loss.backward()
    if (k + 1) % n_accumulation == 0:
        optimizer.step()
        optimizer.zero_grad()
```

**Avoid synchronizations in the training loop.** `loss.item()`, `print(loss)` or
moving a tensor to the CPU force Python to wait for the GPU. Log every few hundred
steps, not at every step.

**Save checkpoints** of the model *and* of the optimizer, to resume a training that
crashed and to keep the best model on the validation set:

```python
torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "epoch": epoch}, "checkpoint.pt")
```

**Log your experiments** with [TensorBoard](https://pytorch.org/docs/stable/tensorboard.html)
or [Weights & Biases](https://wandb.ai): you will not remember which run used which
learning rate.

**Scaling laws.** For large models, the loss decreases as a power law of the model
size, the data size and the compute
([Kaplan et al., 2020](https://arxiv.org/abs/2001.08361)). For a fixed compute
budget, the model size and the number of training tokens should grow together, about
20 tokens per parameter for language models
([Hoffmann et al., 2022](https://arxiv.org/abs/2203.15556)). These laws are fitted on
small runs to choose the size of a large one.